In [24]:
import json
from pathlib import Path
from rdflib import Graph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

from pyeed.core import ProteinRecord

## Get `ProteinRecord` instances

In [3]:
with open("../../docs/examples/ids.json") as f:
    ids = json.load(f)

seqs = ProteinRecord.get_ids(ids)

Output()

In [21]:
# write ttl files
for seq in seqs:
    g = Graph()
    g.parse(data=seq.json(), format="json-ld")
    g.serialize(destination=f"../ttl/{seq.id}.ttl", format="turtle")

## Connect to neo4j database

In [40]:
# set the configuration to connect to your Aura DB
AURA_DB_URI = "bolt://localhost:7687"

auth_data = {
    "uri": AURA_DB_URI,
    "database": "neo4j",
    "user": "None",
    "pwd": "None",
}

# Define your custom mappings & store config
config = Neo4jStoreConfig(
    auth_data=auth_data,
    # custom_prefixes=prefixes,
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=True,
)

# Create the RDF Graph, parse & ingest the data to Neo4j, and close the store
# (If the field batching is set to True in the Neo4jStoreConfig, remember to close the store to prevent
# the loss of any uncommitted records.)
neo4j_aura = Graph(store=Neo4jStore(config=config))
# Calling the parse method will implictly open the store


for path in Path("../ttl").rglob("*.ttl"):
    neo4j_aura.parse(str(path), format="ttl")

neo4j_aura.close(True)

Uniqueness constraint on :Resource(uri) found. 
                
                
IMPORTED 946 TRIPLES
